# Processamento em Lote (Batch Processing) - Tarefa de Decision Making
Iteração sobre a amostra completa de sujeitos (n=54) para extração automática de métricas fisiológicas (GSR e Frequência Cardíaca) e neurofisiológicas (fNIRS). O processamento inclui a deteção de canais ativos, filtragem de sinais e segmentação temporal com base nos marcadores do sistema NIRx (ficheiros `.snirf` e `.wings`). 
No final, é exportado um ficheiro CSV mestre e consolidado com as variáveis fásicas (amplitudes máximas e variação média), com tratamento de exceções (try/except) para evitar a interrupção da rotina por ficheiros em falta.

In [2]:
import os
import zipfile
import mne
import mne_nirs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

# ==========================================
# 1. Configuração do Lote e Parâmetros Base
# ==========================================
BASE_DIR = r"C:\Users\guilh\Desktop\OneDrive - Universidade de Coimbra\PhD_Project_ARDITI_Guilherme Saldanha\Study2_Data Collection Files_PhD_Guilherme Saldanha_UC-UMa-ITI"
NUM_SUJEITOS = 3 
TAREFA = "tk-dm" # Simplifiquei para apanhar qualquer run de Decision Making

# Parâmetros de Filtragem e Epoching (Fisiologia e fNIRS)
LIMIAR_SCI = 0.7
LOW_CUTOFF_HZ = 0.01
HIGH_CUTOFF_HZ = 0.10
PPF = 0.1
DISTANCIA_MIN_LONGA_M = 0.015

JANELA_INICIO_S = -5.0
JANELA_FIM_S = 25.0
TRIGGER_VIDEO = '120'

dados_mestre = []

print(f"A iniciar processamento em lote para {NUM_SUJEITOS} sujeitos (Tarefa: {TAREFA})...\n")

# ==========================================
# 2. Ciclo de Processamento (Iterar sujeitos)
# ==========================================
for i in range(1, NUM_SUJEITOS + 1):
    sujeito_id = f"sub-{i:03d}"
    pasta_nirs = os.path.join(BASE_DIR, sujeito_id, "nirs")
    
    print(f"[{sujeito_id}] A processar...")
    
    try:
        # -------------------------------------------------------------
        # 2.1 Identificação Dinâmica da Pasta da Tarefa
        # -------------------------------------------------------------
        if not os.path.exists(pasta_nirs):
            raise FileNotFoundError(f"Pasta raiz 'nirs' não existe para {sujeito_id}.")
            
        # Procura qualquer pasta dentro de 'nirs' que contenha 'tk-dm'
        pastas_dentro_nirs = os.listdir(pasta_nirs)
        nome_pasta_run = next((p for p in pastas_dentro_nirs if TAREFA in p and os.path.isdir(os.path.join(pasta_nirs, p))), None)
        
        if not nome_pasta_run:
            raise FileNotFoundError(f"Nenhuma pasta contendo a tarefa '{TAREFA}' foi encontrada.")
            
        pasta_alvo = os.path.join(pasta_nirs, nome_pasta_run)
        
        # -------------------------------------------------------------
        # 2.2 Identificação dos ficheiros (.snirf, .zip, .wings)
        # -------------------------------------------------------------
        ficheiros_pasta = os.listdir(pasta_alvo)
        snirf_file = next((f for f in ficheiros_pasta if f.endswith('.snirf')), None)
        zip_file = next((f for f in ficheiros_pasta if f.endswith('.zip')), None)
        
        if not snirf_file or not zip_file:
            raise FileNotFoundError("Ficheiro .snirf ou .zip não encontrados.")
            
        caminho_snirf = os.path.join(pasta_alvo, snirf_file)
        caminho_zip = os.path.join(pasta_alvo, zip_file)
        pasta_extraida_wings = os.path.join(pasta_alvo, "dados_wings_extraidos")
        
        # Extrair dados WINGS se a pasta ainda não existir
        if not os.path.exists(pasta_extraida_wings):
            with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
                zip_ref.extractall(pasta_extraida_wings)
                
        ficheiros_extraidos = os.listdir(pasta_extraida_wings)
        wings_file = next((f for f in ficheiros_extraidos if f.endswith('.wings')), None)
        
        if not wings_file:
             raise FileNotFoundError("Ficheiro .wings não encontrado na pasta extraída.")
             
        caminho_wings = os.path.join(pasta_extraida_wings, wings_file)

        # -------------------------------------------------------------
        # 2.3 Sincronização e Extração de Triggers (fNIRS -> WINGS)
        # -------------------------------------------------------------
        raw_temp = mne.io.read_raw_snirf(caminho_snirf, preload=False, verbose=False)
        
        tempos_video = []
        for onset, duration, desc in zip(raw_temp.annotations.onset, raw_temp.annotations.duration, raw_temp.annotations.description):
            if desc == TRIGGER_VIDEO:
                tempos_video.append(onset)
                
        if len(tempos_video) == 0:
            print(f"  [AVISO] {sujeito_id}: Nenhum marcador 'Video' ({TRIGGER_VIDEO}) encontrado no fNIRS.")
            continue
            
        df_wings = pd.read_csv(caminho_wings, sep=';', skiprows=1)
        df_wings.columns = df_wings.columns.str.strip()
        
        tempos_wings = df_wings['SecondsSinceRunStart'].values
        fs_wings = len(tempos_wings) / tempos_wings[-1]
        
        indices_video = []
        for t in tempos_video:
            idx_mais_proximo = np.argmin(np.abs(tempos_wings - t))
            indices_video.append(idx_mais_proximo)

        # -------------------------------------------------------------
        # 2.4 Processamento Fisiológico (GSR)
        # -------------------------------------------------------------
        gsr_bruto = df_wings['GSR'].values
        
        cutoff_low = 5.0
        nyq = 0.5 * fs_wings
        normal_cutoff = cutoff_low / nyq
        b, a = butter(4, normal_cutoff, btype='low', analog=False)
        gsr_filtrado = filtfilt(b, a, gsr_bruto)
        
        amostras_antes = int(abs(JANELA_INICIO_S) * fs_wings)
        amostras_depois = int(JANELA_FIM_S * fs_wings)
        
        epocas_gsr = []
        for idx_inicio in indices_video:
            idx_start = idx_inicio - amostras_antes
            idx_end = idx_inicio + amostras_depois
            
            if idx_start >= 0 and idx_end < len(gsr_filtrado):
                sinal = gsr_filtrado[idx_start:idx_end]
                baseline = np.mean(sinal[0:amostras_antes])
                epocas_gsr.append(sinal - baseline)
                
        gsr_medio = np.mean(np.array(epocas_gsr), axis=0)

        # -------------------------------------------------------------
        # 2.5 Processamento Fisiológico (Frequência Cardíaca - HR)
        # -------------------------------------------------------------
        hr_bruto = df_wings['HR'].values
        
        epocas_hr = []
        for idx_inicio in indices_video:
            idx_start = idx_inicio - amostras_antes
            idx_end = idx_inicio + amostras_depois
            
            if idx_start >= 0 and idx_end < len(hr_bruto):
                sinal = hr_bruto[idx_start:idx_end]
                baseline = np.mean(sinal[0:amostras_antes])
                epocas_hr.append(sinal - baseline)
                
        hr_medio = np.mean(np.array(epocas_hr), axis=0)

        # -------------------------------------------------------------
        # 2.6 Extração das Métricas para a Base de Dados
        # -------------------------------------------------------------
        gsr_media_epoca = np.mean(gsr_medio)
        gsr_amp_max = np.max(gsr_medio) 
        
        hr_media_epoca = np.mean(hr_medio)
        hr_bradicardia_max = np.min(hr_medio)
        
        registo = {
            'ID': sujeito_id,
            'Tarefa': 'Decision_Making',
            'GSR_Media': gsr_media_epoca,
            'GSR_Amplitude_Max': gsr_amp_max,
            'HR_Media': hr_media_epoca,
            'HR_Bradicardia_Max': hr_bradicardia_max
        }
        
        dados_mestre.append(registo)
        print(f"  [OK] Processado com sucesso ({len(tempos_video)} vídeos).")
        
    except Exception as e:
        print(f"  [ERRO] Falha no processamento: {e}")

# ==========================================
# 3. Exportação da Base de Dados Consolidada
# ==========================================
if len(dados_mestre) > 0:
    df_mestre = pd.DataFrame(dados_mestre)
    
    pasta_exportacao = os.path.join(BASE_DIR, "derivatives", "mne-nirs")
    os.makedirs(pasta_exportacao, exist_ok=True)
    caminho_csv_final = os.path.join(pasta_exportacao, "dataset_mestre_fisio_DM.csv")
    
    df_mestre.to_csv(caminho_csv_final, index=False)
    
    print(f"\nProcessamento em Lote Concluído! Dados válidos: {len(dados_mestre)}")
    print(f"Base de dados exportada para:\n{caminho_csv_final}")
else:
    print("\nProcessamento concluído, mas nenhum sujeito obteve dados válidos para exportação.")

A iniciar processamento em lote para 3 sujeitos (Tarefa: tk-dm)...

[sub-001] A processar...
  [OK] Processado com sucesso (10 vídeos).
[sub-002] A processar...
  [OK] Processado com sucesso (10 vídeos).
[sub-003] A processar...
  [OK] Processado com sucesso (10 vídeos).

Processamento em Lote Concluído! Dados válidos: 3
Base de dados exportada para:
C:\Users\guilh\Desktop\OneDrive - Universidade de Coimbra\PhD_Project_ARDITI_Guilherme Saldanha\Study2_Data Collection Files_PhD_Guilherme Saldanha_UC-UMa-ITI\derivatives\mne-nirs\dataset_mestre_fisio_DM.csv


# Processamento em Lote (Batch) - Sinais Periféricos (GSR e HR) | Tarefa: Decision Making
Extração em lote das métricas fisiológicas fásicas (GSR e HR) para a amostra (n=54). 
Inclui a sincronização temporal via `.snirf`, a segmentação por épocas centradas no estímulo "Vídeo" e o cálculo da amplitude máxima (suor) e da bradicardia máxima (desaceleração cardíaca). 
**Controlo de Qualidade:** Sujeitos com falhas de hardware identificadas (ex: erro de PPG/SPO2 no sub-042) têm as métricas correspondentes automaticamente codificadas como `NaN` (Missing Value) para preservar a integridade estatística, mantendo os restantes sinais intactos.

In [1]:
import os
import zipfile
import mne
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import warnings

# Ignorar os avisos do MNE para não poluir o terminal durante o loop de 54 sujeitos
warnings.filterwarnings('ignore')

# ==========================================
# 1. Configuração do Lote e Parâmetros Base
# ==========================================
BASE_DIR = r"C:\Users\guilh\Desktop\OneDrive - Universidade de Coimbra\PhD_Project_ARDITI_Guilherme Saldanha\Study2_Data Collection Files_PhD_Guilherme Saldanha_UC-UMa-ITI"
NUM_SUJEITOS = 54 
TAREFA = "tk-dm" 

JANELA_INICIO_S = -5.0
JANELA_FIM_S = 25.0
TRIGGER_VIDEO = '120'

# Listas de Exclusão (Blacklists) por falha de hardware
SUJEITOS_SEM_HR = ['sub-042'] # Falha no sensor PPG
SUJEITOS_SEM_GSR = []         # Adicionar IDs se houver falhas exclusivas de GSR

dados_mestre = []

print(f"A iniciar processamento periférico em lote para {NUM_SUJEITOS} sujeitos...\n")

# ==========================================
# 2. Ciclo de Processamento
# ==========================================
for i in range(1, NUM_SUJEITOS + 1):
    sujeito_id = f"sub-{i:03d}"
    pasta_nirs = os.path.join(BASE_DIR, sujeito_id, "nirs")
    
    try:
        if not os.path.exists(pasta_nirs):
            raise FileNotFoundError("Pasta 'nirs' não existe.")
            
        pastas_dentro_nirs = os.listdir(pasta_nirs)
        nome_pasta_run = next((p for p in pastas_dentro_nirs if TAREFA in p and os.path.isdir(os.path.join(pasta_nirs, p))), None)
        
        if not nome_pasta_run:
            raise FileNotFoundError(f"Pasta da tarefa '{TAREFA}' não encontrada.")
            
        pasta_alvo = os.path.join(pasta_nirs, nome_pasta_run)
        
        ficheiros_pasta = os.listdir(pasta_alvo)
        snirf_file = next((f for f in ficheiros_pasta if f.endswith('.snirf')), None)
        zip_file = next((f for f in ficheiros_pasta if f.endswith('.zip')), None)
        
        if not snirf_file or not zip_file:
            raise FileNotFoundError("Ficheiros .snirf ou .zip em falta.")
            
        caminho_snirf = os.path.join(pasta_alvo, snirf_file)
        caminho_zip = os.path.join(pasta_alvo, zip_file)
        pasta_extraida_wings = os.path.join(pasta_alvo, "dados_wings_extraidos")
        
        if not os.path.exists(pasta_extraida_wings):
            with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
                zip_ref.extractall(pasta_extraida_wings)
                
        ficheiros_extraidos = os.listdir(pasta_extraida_wings)
        wings_file = next((f for f in ficheiros_extraidos if f.endswith('.wings')), None)
        
        if not wings_file:
             raise FileNotFoundError("Ficheiro .wings não extraído.")
             
        caminho_wings = os.path.join(pasta_extraida_wings, wings_file)

        # -- Sincronização (.snirf) --
        raw_temp = mne.io.read_raw_snirf(caminho_snirf, preload=False, verbose=False)
        tempos_video = [onset for onset, desc in zip(raw_temp.annotations.onset, raw_temp.annotations.description) if desc == TRIGGER_VIDEO]
                
        if len(tempos_video) == 0:
            print(f"  [{sujeito_id}] AVISO: Nenhum marcador de Vídeo. A saltar...")
            continue
            
        df_wings = pd.read_csv(caminho_wings, sep=';', skiprows=1)
        df_wings.columns = df_wings.columns.str.strip()
        
        tempos_wings = df_wings['SecondsSinceRunStart'].values
        fs_wings = len(tempos_wings) / tempos_wings[-1]
        
        indices_video = [np.argmin(np.abs(tempos_wings - t)) for t in tempos_video]
        amostras_antes = int(abs(JANELA_INICIO_S) * fs_wings)
        amostras_depois = int(JANELA_FIM_S * fs_wings)

        # -- Processamento GSR --
        if sujeito_id not in SUJEITOS_SEM_GSR:
            gsr_bruto = df_wings['GSR'].values
            nyq = 0.5 * fs_wings
            b, a = butter(4, 5.0 / nyq, btype='low', analog=False)
            gsr_filtrado = filtfilt(b, a, gsr_bruto)
            
            epocas_gsr = []
            for idx_inicio in indices_video:
                if idx_inicio - amostras_antes >= 0 and idx_inicio + amostras_depois < len(gsr_filtrado):
                    sinal = gsr_filtrado[idx_inicio - amostras_antes : idx_inicio + amostras_depois]
                    epocas_gsr.append(sinal - np.mean(sinal[0:amostras_antes]))
                    
            gsr_medio = np.mean(np.array(epocas_gsr), axis=0)
            gsr_media_epoca = np.mean(gsr_medio)
            gsr_amp_max = np.max(gsr_medio)
        else:
            gsr_media_epoca = np.nan
            gsr_amp_max = np.nan

        # -- Processamento HR --
        if sujeito_id not in SUJEITOS_SEM_HR:
            hr_bruto = df_wings['HR'].values
            
            epocas_hr = []
            for idx_inicio in indices_video:
                if idx_inicio - amostras_antes >= 0 and idx_inicio + amostras_depois < len(hr_bruto):
                    sinal = hr_bruto[idx_inicio - amostras_antes : idx_inicio + amostras_depois]
                    epocas_hr.append(sinal - np.mean(sinal[0:amostras_antes]))
                    
            hr_medio = np.mean(np.array(epocas_hr), axis=0)
            hr_media_epoca = np.mean(hr_medio)
            hr_bradicardia_max = np.min(hr_medio)
        else:
            hr_media_epoca = np.nan
            hr_bradicardia_max = np.nan

        # -- Adicionar à BD --
        dados_mestre.append({
            'ID': sujeito_id,
            'Tarefa': 'Decision_Making',
            'GSR_Media': gsr_media_epoca,
            'GSR_Amplitude_Max': gsr_amp_max,
            'HR_Media': hr_media_epoca,
            'HR_Bradicardia_Max': hr_bradicardia_max
        })
        print(f"  [{sujeito_id}] OK ({len(tempos_video)} vídeos).")
        
    except Exception as e:
        print(f"  [{sujeito_id}] ERRO: {e}")

# ==========================================
# 3. Exportação da Base de Dados Consolidada
# ==========================================
if dados_mestre:
    df_mestre = pd.DataFrame(dados_mestre)
    pasta_exportacao = os.path.join(BASE_DIR, "derivatives", "mne-nirs")
    os.makedirs(pasta_exportacao, exist_ok=True)
    caminho_csv_final = os.path.join(pasta_exportacao, "dataset_mestre_fisio_DM.csv")
    df_mestre.to_csv(caminho_csv_final, index=False)
    print(f"\nProcessamento concluído! Dados extraídos de {len(dados_mestre)} sujeitos.")

A iniciar processamento periférico em lote para 54 sujeitos...

  [sub-001] OK (10 vídeos).
  [sub-002] OK (10 vídeos).
  [sub-003] OK (10 vídeos).
  [sub-004] OK (10 vídeos).
  [sub-005] OK (10 vídeos).
  [sub-006] OK (10 vídeos).
  [sub-007] OK (10 vídeos).
  [sub-008] OK (10 vídeos).
  [sub-009] OK (10 vídeos).
  [sub-010] OK (10 vídeos).
  [sub-011] OK (10 vídeos).
  [sub-012] OK (10 vídeos).
  [sub-013] OK (10 vídeos).
  [sub-014] OK (10 vídeos).
  [sub-015] OK (7 vídeos).
  [sub-016] OK (10 vídeos).
  [sub-017] OK (10 vídeos).
  [sub-018] OK (10 vídeos).
  [sub-019] OK (10 vídeos).
  [sub-020] OK (10 vídeos).
  [sub-021] OK (10 vídeos).
  [sub-022] OK (10 vídeos).
  [sub-023] OK (10 vídeos).
  [sub-024] OK (10 vídeos).
  [sub-025] OK (10 vídeos).
  [sub-026] OK (10 vídeos).
  [sub-027] OK (10 vídeos).
  [sub-028] OK (10 vídeos).
  [sub-029] OK (10 vídeos).
  [sub-030] OK (10 vídeos).
  [sub-031] OK (10 vídeos).
  [sub-032] OK (10 vídeos).
  [sub-033] OK (10 vídeos).
  [sub-034] O